In [1]:
import pandas as pd
import altair as alt
from pathlib import Path

In [3]:
file_path = Path("../data/eic2025_genero.xlsx")

df_raw = pd.read_excel(
    file_path,
    sheet_name="12",
    header=None
)

labor = df_raw.loc[
    (df_raw[0] == "Estados Unidos Mexicanos") &
    (df_raw[2] == "Valor") &
    (df_raw[1] != "Total"),
    [1, 7, 8]
].copy()

labor.columns = ["age_group", "Women", "Men"]

labor[["Women", "Men"]] = labor[["Women", "Men"]].apply(
    pd.to_numeric, errors="coerce"
)

display(labor)

,age_group,Women,Men
14,12-17 años,5.459109,14.569907
19,18-29 años,46.513133,75.051754
24,30-59 años,55.986425,92.977211
29,60-64 años,29.409825,65.690363
34,65 años y más,11.964152,35.212353


In [5]:
labor_long = labor.melt(
    id_vars="age_group",
    var_name="sex",
    value_name="participation_rate"
)

labor_long["age_group"] = labor_long["age_group"].replace({
    "12-17 años": "12–17",
    "18-29 años": "18–29",
    "30-59 años": "30–59",
    "60-64 años": "60–64",
    "65 años y más": "65 and older"
})

age_order = ["12–17", "18–29", "30–59", "60–64", "65+"]

display(labor_long)

,age_group,sex,participation_rate
0,12–17,Women,5.459109
1,18–29,Women,46.513133
2,30–59,Women,55.986425
3,60–64,Women,29.409825
4,65 and older,Women,11.964152
5,12–17,Men,14.569907
6,18–29,Men,75.051754
7,30–59,Men,92.977211
8,60–64,Men,65.690363
9,65 and older,Men,35.212353


In [14]:
base = alt.Chart(labor_long).encode(
    y=alt.Y(
        "age_group:N",
        sort=age_order,
        title="Age group",
        axis=alt.Axis(
            labelFontSize=12,
            grid=False
        )
    ),
    x=alt.X(
        "participation_rate:Q",
        title="Labor force participation rate (%)",
        scale=alt.Scale(domain=[0, 100]),
        axis=alt.Axis(
            values=[0, 20, 40, 60, 80, 100],
            grid=True,
            gridOpacity=0.55
        )
    )
)


# Connecting lines
lines = base.mark_line(
    color="#B0B0B0",
    strokeWidth=3
).encode(
    detail="age_group:N"
)

# Points
points = base.mark_circle(size=170).encode(
    color=alt.Color(
        "sex:N",
        scale=alt.Scale(
            domain=["Women", "Men"],
            range=["#8624F5", "#1FC3AA"]
        ),
        legend=alt.Legend(
            title=None,
            orient="top-right"
        )
    )
)

# Labels for women
labels_women = (
    base
    .transform_filter(alt.datum.sex == "Women")
    .mark_text(
        align="right",
        dx=-12,
        fontSize=12
    )
    .encode(
        text=alt.Text("participation_rate:Q", format=".1f")
    )
)

# Labels for men
labels_men = (
    base
    .transform_filter(alt.datum.sex == "Men")
    .mark_text(
        align="left",
        dx=12,
        fontSize=12
    )
    .encode(
        text=alt.Text("participation_rate:Q", format=".1f")
    )
)

gap_annotation = (
    alt.Chart(pd.DataFrame({
        "age_group": ["30–59"],
        "participation_rate": [74.5],
        "label": ["37.0 pp gap"]
    }))
    .mark_text(
        dy=-18,
        fontSize=12,
        fontWeight="bold",
        color="#D97706"
    )
    .encode(
        x="participation_rate:Q",
        y=alt.Y("age_group:N", sort=age_order),
        text="label:N"
    )
)

chart_labor = (
    (lines + points + labels_women + labels_men + gap_annotation)
    .properties(
        width=650,
        height=300,
        title=alt.TitleParams(
            text="Gender Gaps in Labor Force Participation",
            subtitle="Mexico, 2025",
            anchor="middle"
        )
    )
    .configure_view(stroke=None)
)

chart_labor

alt.LayerChart(...)

In [15]:
chart_labor.save("../images/labor_force_participation.png", scale_factor=2)

In [16]:
heatmap_data = df_raw.loc[
    (df_raw[2] == "Valor") &
    (df_raw[1] != "Total") &
    (df_raw[0] != "Estados Unidos Mexicanos"),
    [0, 1, 7, 8]
].copy()

heatmap_data.columns = [
    "state", "age_group", "Women", "Men"
]

heatmap_data[["Women", "Men"]] = heatmap_data[
    ["Women", "Men"]
].apply(pd.to_numeric, errors="coerce")

# Gender gap in percentage points
heatmap_data["gender_gap"] = (
    heatmap_data["Men"] - heatmap_data["Women"]
)

# Remove INEGI's numeric state codes
heatmap_data["state"] = (
    heatmap_data["state"]
    .str.replace(r"^\d{2}\s+", "", regex=True)
    .str.strip()
)

# Translate age groups
heatmap_data["age_group"] = heatmap_data["age_group"].replace({
    "12-17 años": "12–17",
    "18-29 años": "18–29",
    "30-59 años": "30–59",
    "60-64 años": "60–64",
    "65 años y más": "65+"
})

display(heatmap_data.head())
print("States:", heatmap_data["state"].nunique())
print("Observations:", len(heatmap_data))

,state,age_group,Women,Men,gender_gap
44,Aguascalientes,12–17,6.161886,14.673046,8.511160
49,Aguascalientes,18–29,53.590256,77.670837,24.080581
54,Aguascalientes,30–59,60.042579,93.654682,33.612103
59,Aguascalientes,60–64,29.528184,61.536138,32.007954
64,Aguascalientes,65+,10.276113,30.692659,20.416546


States: 32
Observations: 160


In [17]:
state_order = (
    heatmap_data[heatmap_data["age_group"] == "30–59"]
    .sort_values("gender_gap", ascending=False)["state"]
    .tolist()
)

age_order = ["12–17", "18–29", "30–59", "60–64", "65+"]

heatmap = (
    alt.Chart(heatmap_data)
    .mark_rect(stroke="white", strokeWidth=1)
    .encode(
        x=alt.X(
            "age_group:N",
            sort=age_order,
            title="Age group",
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "state:N",
            sort=state_order,
            title=None
        ),
        color=alt.Color(
            "gender_gap:Q",
            title="Gender gap (pp)",
            scale=alt.Scale(
                scheme="purples"
            )
        ),
        tooltip=[
            alt.Tooltip("state:N", title="State"),
            alt.Tooltip("age_group:N", title="Age group"),
            alt.Tooltip("Women:Q", title="Women (%)", format=".1f"),
            alt.Tooltip("Men:Q", title="Men (%)", format=".1f"),
            alt.Tooltip("gender_gap:Q", title="Gender gap (pp)", format=".1f")
        ]
    )
    .properties(
        width=600,
        height=750,
        title=alt.TitleParams(
            text="Gender Gaps in Labor Force Participation Across Mexico",
            subtitle="Difference between men's and women's participation rates, 2025",
            anchor="middle"
        )
    )
    .configure_view(stroke=None)
)

heatmap

alt.Chart(...)

In [18]:
df_13 = pd.read_excel(
    file_path,
    sheet_name="13",
    header=None
)

display(df_13.iloc[5:12, :])

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
5,Entidad federativa,Grupos de edad,Estimador,Población de 12 años y más no económicamente a...,NaN,NaN,Tipo de actividad no económica,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,Personas pensionadas o jubiladas,NaN,NaN,NaN,...,NaN,NaN,Personas con alguna limitación física o mental...,NaN,NaN,NaN,Personas en otras actividades no económicas,NaN,NaN,NaN
7,NaN,NaN,NaN,Total,Mujeres,Hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres,...,Hombres,Diferencia mujeres-hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Estados Unidos Mexicanos,Total,Valor,48541538,33497815,15043723,10.850892,6.350766,20.87131,-14.520543,...,5.372925,56.127214,3.410916,2.156436,6.204262,-4.047826,9.286657,5.694258,17.285841,-11.591583
10,Estados Unidos Mexicanos,Total,Error estándar,184999.322604,125550.205852,63649.014555,0.059284,0.050159,0.098865,0.110862,...,0.042469,0.089375,0.019359,0.016174,0.040028,0.043173,0.055581,0.051778,0.088071,0.102164
11,Estados Unidos Mexicanos,Total,Límite inferior de confianza,48237240.76653,33291302.998958,14939029.54075,10.753763,6.268761,20.709156,-14.702894,...,5.303497,55.980206,3.379216,2.129992,6.138746,-4.118839,9.195637,5.609687,17.141456,-11.759628


In [19]:
for i in range(df_13.shape[1]):
    print(
        i,
        "|",
        df_13.iloc[5, i],
        "|",
        df_13.iloc[6, i]
    )

0 | Entidad federativa | nan
1 | Grupos de edad | nan
2 | Estimador | nan
3 | Población de 12 años y más no económicamente activa | nan
4 | nan | nan
5 | nan | nan
6 | Tipo de actividad no económica | Personas pensionadas o jubiladas
7 | nan | nan
8 | nan | nan
9 | nan | nan
10 | nan | Personas estudiantes
11 | nan | nan
12 | nan | nan
13 | nan | nan
14 | nan | Personas dedicadas a los quehaceres de su hogar
15 | nan | nan
16 | nan | nan
17 | nan | nan
18 | nan | Personas con alguna limitación física o mental que les impide trabajar
19 | nan | nan
20 | nan | nan
21 | nan | nan
22 | nan | Personas en otras actividades no económicas
23 | nan | nan
24 | nan | nan
25 | nan | nan


In [22]:
# National data, all ages, estimated values
national = df_13.loc[
    (df_13[0] == "Estados Unidos Mexicanos") &
    (df_13[1] == "Total") &
    (df_13[2] == "Valor")
]

activities = {
    "Retired or pensioners": (7, 8),
    "Students": (11, 12),
    "Household duties": (15, 16),
    "Unable to work": (19, 20),
    "Other activities": (23, 24)
}

records = []

for activity, (women_col, men_col) in activities.items():
    records.append({
        "activity": activity,
        "sex": "Women",
        "percentage": float(national.iloc[0, women_col])
    })
    records.append({
        "activity": activity,
        "sex": "Men",
        "percentage": float(national.iloc[0, men_col])
    })

inactive_long = pd.DataFrame(records)

display(inactive_long)

,activity,sex,percentage
0,Retired or pensioners,Women,6.350766
1,Retired or pensioners,Men,20.871310
2,Students,Women,24.298400
3,Students,Men,50.265662
4,Household duties,Women,61.500140
5,Household duties,Men,5.372925
6,Unable to work,Women,2.156436
7,Unable to work,Men,6.204262
8,Other activities,Women,5.694258
9,Other activities,Men,17.285841


In [26]:
inactive_long.groupby("sex")["percentage"].sum()

inactive_long["activity"] = inactive_long["activity"].replace({
    "Retired or pensioners": "Retired",
    "Other activities": "Other"
})

In [27]:
activity_order = [
    "Household duties",
    "Students",
    "Retired",
    "Unable to work",
    "Other"
]

activity_colors = [
    "#8624F5",
    "#1FC3AA",
    "#7895B2",
    "#E5A34D",
    "#C7CBD1"
]

chart_inactive = (
    alt.Chart(inactive_long)
    .mark_bar(size=55)
    .encode(
        y=alt.Y(
            "sex:N",
            sort=["Women", "Men"],
            title=None,
            axis=alt.Axis(labelFontSize=13)
        ),
        x=alt.X(
            "percentage:Q",
            stack="zero",
            title="Share of population outside the labor force (%)",
            scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(values=[0, 20, 40, 60, 80, 100])
        ),
        color=alt.Color(
            "activity:N",
            scale=alt.Scale(
                domain=activity_order,
                range=activity_colors
            ),
            legend=alt.Legend(
                title=None,
                orient="bottom",
                columns=3
            )
        ),
        order=alt.Order(
            "activity:N",
            sort="ascending"
        ),
        tooltip=[
            alt.Tooltip("sex:N", title="Sex"),
            alt.Tooltip("activity:N", title="Activity"),
            alt.Tooltip("percentage:Q", title="Share (%)", format=".1f")
        ]
    )
    .properties(
        width=700,
        height=180,
        title=alt.TitleParams(
            text="Distribution of the economically inactive population by reported activity",
            subtitle="Mexico, 2025",
            anchor="middle"
        )
    )
    .configure_view(stroke=None)
)

chart_inactive

alt.Chart(...)

In [28]:
chart_inactive.save("../images/inactive_population_activities.png", scale_factor=2)